In [1]:
import numpy as np
import random
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
import tensorflow as tf

RANDOM_SEED = 123
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [2]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()

x_train = x_train.astype('float32') /255.0
x_test = x_test.astype('float32') /255.0

x_train = np.expand_dims(x_train, -1)
x_test = np.expand_dims(x_test, -1)

num_classes = 10

y_train_cat = keras.utils.to_categorical(y_train, 10)
y_test_cat = keras.utils.to_categorical(y_test, 10)


11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
model = keras.models.Sequential([
        layers.Input(shape = (28, 28, 1)),
        layers.Conv2D(32, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),

        layers.Conv2D(64, (3,3), activation='relu', padding='same'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(10, activation='softmax', name='classifier')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.fit(x_train, y_train_cat, epochs=10)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 113s 59ms/step - accuracy: 0.8948 - loss: 0.3436
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9826 - loss: 0.0560
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 82s 41ms/step - accuracy: 0.9878 - loss: 0.0396
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 78s 42ms/step - accuracy: 0.9909 - loss: 0.0287
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 80s 41ms/step - accuracy: 0.9933 - loss: 0.0207
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 77s 41ms/step - accuracy: 0.9947 - loss: 0.0159
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 77s 41ms/step - accuracy: 0.9963 - loss: 0.0124
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 77s 41ms/step - accuracy: 0.9971 - loss: 0.0099
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 81s 41ms/step - accuracy: 0.9976 - loss: 0.0075
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 85s 42ms/step - accuracy: 0.9977 - loss: 0.0067


In [14]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

def evaluate_classification(y_true, y_pred, output_prefix="model_eval"):

    # =========================
    # Preparación de etiquetas
    # =========================
    y_pred_classes = np.argmax(y_pred, axis=1)
    y_true_flat = y_true.reshape(-1)

    num_classes = y_pred.shape[1]

    # =========================
    # Métricas principales
    # =========================
    accuracy = accuracy_score(y_true_flat, y_pred_classes)
    precision_macro = precision_score(y_true_flat, y_pred_classes, average="macro")
    recall_macro = recall_score(y_true_flat, y_pred_classes, average="macro")
    f1_macro = f1_score(y_true_flat, y_pred_classes, average="macro")

    loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
    loss = loss_fn(y_true, y_pred).numpy()

    top5 = tf.keras.metrics.SparseTopKCategoricalAccuracy(k=min(5, num_classes))
    top5.update_state(y_true, y_pred)
    top5_acc = top5.result().numpy()

    # =========================
    # Guardar métricas en CSV
    # =========================
    metrics_dict = {
        "Accuracy": accuracy,
        "Precision_macro": precision_macro,
        "Recall_macro": recall_macro,
        "F1_macro": f1_macro,
        "Loss": loss,
        "Top5_accuracy": top5_acc
    }

    df_metrics = pd.DataFrame([metrics_dict])
    df_metrics.to_csv(f"{output_prefix}_metrics.csv", index=False)

    # =========================
    # Classification Report
    # =========================
    report = classification_report(y_true_flat, y_pred_classes, output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    df_report.to_csv(f"{output_prefix}_classification_report.csv")

    # =========================
    # Matriz de Confusión
    # =========================
    cm = confusion_matrix(y_true_flat, y_pred_classes)

    plt.figure(figsize=(8,6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_confusion_matrix.svg", format="svg")
    plt.close()

    # =========================
    # Matriz Normalizada
    # =========================
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

    plt.figure(figsize=(8,6))
    sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues")
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title("Normalized Confusion Matrix")
    plt.tight_layout()
    plt.savefig(f"{output_prefix}_confusion_matrix_normalized.svg", format="svg")
    plt.close()

    plt.figure(figsize=(10,4))

    for i in range(10):
        plt.subplot(2,5,i+1)
        plt.imshow(predict_x[i])
        plt.title(f"True: {y_true_flat[i]}\nPred: {y_pred_classes[i]}")
        plt.axis("off")

    plt.tight_layout()
    plt.savefig(f"{output_prefix}_ten_images.svg", format="svg")
    plt.close()

    print("Evaluación completada y archivos guardados.")
    print(metrics_dict)

    return metrics_dict


In [19]:
predict_x = x_train[0:100]
y_true =  y_train[0:100]

y_pred = model(predict_x, training=False).numpy()


In [20]:
y_pred.shape

(100, 10)

In [21]:
evaluate_classification(y_true, y_pred, output_prefix="CNN_MLP")

Evaluación completada y archivos guardados.
{'Accuracy': 0.99, 'Precision_macro': 0.9888888888888889, 'Recall_macro': 0.9909090909090909, 'F1_macro': 0.9893557422969188, 'Loss': np.float32(0.011652803), 'Top5_accuracy': np.float32(1.0)}


{'Accuracy': 0.99,
 'Precision_macro': 0.9888888888888889,
 'Recall_macro': 0.9909090909090909,
 'F1_macro': 0.9893557422969188,
 'Loss': np.float32(0.011652803),
 'Top5_accuracy': np.float32(1.0)}

In [22]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ classifier (Dense)              │ (None, 10)             │        31,370 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 150,560 (588.13 KB)

 Trainable params: 50,186 (196.04 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 100,374 (392.09 KB)